# 3. Funciones generales útiles

In [ ]:
# Librerías
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
def plot_histogram_rgb(image, vis = False):
    '''
    Función que permite graficar los histogramas de las componentes RGB de una imagen.

    Args:
        image: imagen a procesar.
    '''
    # Cálculo de los histogramas
    hist_r = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist_g = cv2.calcHist([image], [1], None, [256], [0, 256])
    hist_b = cv2.calcHist([image], [2], None, [256], [0, 256])

    # Graficar cada uno de los histogramas
    if vis:
        fig, ax = plt.subplots(1, 3, figsize=(30, 10))
        ax[0].plot(hist_r.flatten(), color='red')
        ax[0].set_title('Histograma Rojo')
        ax[0].set_xlabel('Intensidad de iluminación')
        ax[0].set_ylabel('Cantidad de pixeles')
        ax[0].set_xlim([0, 256])

        ax[1].plot(hist_g.flatten(), color='green')
        ax[1].set_title('Histograma Verde')
        ax[1].set_xlabel('Intensidad de iluminación')
        ax[1].set_ylabel('Cantidad de pixeles')
        ax[1].set_xlim([0, 256])

        ax[2].plot(hist_b.flatten(), color='blue')
        ax[2].set_title('Histograma Azul')
        ax[2].set_xlabel('Intensidad de iluminación')
        ax[2].set_ylabel('Cantidad de pixeles')
        ax[2].set_xlim([0, 256])

        plt.show()

    return hist_r, hist_g, hist_b

def saturated_histogram(array_image):
    '''
    Función que permite saturar los valores de un histograma, siendo la distribución:
    - 0 si el valor es menor a 0.
    - 255 si el valor es mayor a 255.

    Args:
        array_image: arreglo de la imagen a procesar.

    Returns:
        array_image: arreglo de la imagen con los valores saturados.
    '''
    array_image = array_image.astype(np.uint8)

    # Saturación de los valores
    array_image[array_image < 0] = 0
    array_image[array_image > 255] = 255

    return array_image

# 4. Mejora usando Modelos Clásicos

In [ ]:
def contrast_extend(image, channel, lim_a, lim_b):
    '''
    Función que permite extender el contraste de una imagen.

    Args:
        image: imagen a procesar.
        channel: canal de la imagen a procesar.
        lim_a: límite inferior.
        lim_b: límite superior.

    Returns:
        image: imagen con el contraste extendido.
    '''
    # Estiramiento lineal de contraste
    image[:, :, channel] = np.clip((image[:, :, channel] - lim_a) * (255 / (lim_b - lim_a)), 0, 255)

    # Asegurarse de que el tipo de dato sea uint8
    image[:, :, channel] = image[:, :, channel].astype(np.uint8)

    return image

def equal_hist(image):
    '''
    Función que se encarga de la ecualización del histograma
    de una imagen de entrada, aplicando una look-up table.

    Args:
        image: imagen a procesar.

    Returns:
        image_eq: imagen con el histograma ecualizado.
    '''
    # Creación de una copia para preservar la original
    image_eq = image.copy()
    L = 256

    # Ecualización de cada canal por separado
    for channel in range(3):
        # Cálculo del histograma y su acumulado para el canal
        hist = cv2.calcHist([image], [channel], None, [L], [0, L])
        cdf = hist.cumsum()  # Función de distribución acumulativa

        # Normalización de cdf
        cdf_max = cdf.max()
        cdf_min = cdf.min()
        cdf_normalized = (cdf - cdf_min) * (L - 1) / (cdf_max- cdf_min)
        cdf_normalized = cdf_normalized.astype('uint8')  # Conversión a enteros

        # Mapear los valores antiguos a los nuevos usando la look-up table
        image_eq[:, :, channel] = cdf_normalized[image[:, :, channel]]

    return image_eq

def clahe_enhancement(image, clahe_rgb = False):
    '''
    Función que permite realizar la mejora de contraste de una imagen
    mediante CLAHE.

    Args:
        image: imagen a procesar cargada en RGB.
        clahe_rgb: booleano que indica si se aplica CLAHE en espacio rgb
        (todos los canales) o en el espacio HSV (solo el canal Value).

    Returns:
        image_clahe: imagen con el contraste mejorado.
    '''
    # Crear una copia de la imagen sobre la cual trabajar
    image_clahe = image.copy()

    # Crear estructura CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    if clahe_rgb:
        # Aplicar CLAHE a cada canal por separado
        for channel in range(3):
            image_clahe[:, :, channel] = clahe.apply(image_clahe[:, :, channel])

    else:
        # Transformar imagen al dominio HSV
        image_clahe = cv2.cvtColor(image_clahe, cv2.COLOR_RGB2HSV)

        # Aplicar CLAHE al tercer canal (canal Value)
        image_clahe[:, :, 2] = clahe.apply(image_clahe[:, :, 2])

        # Transformar imagen de vuelta al dominio RGB
        image_clahe = cv2.cvtColor(image_clahe, cv2.COLOR_HSV2RGB)

    image_clahe = image_clahe.astype(np.uint8)
    return image_clahe

# 5. Mejora usando *Bread*

# 6. Evaluación y Comparación de Métodos Implementados

# 7. Aplicación Real